🥈 **Camada Silver**

Criação do banco de dados:

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS silver;

Criação da tabela silver.dim_consumidores:

In [0]:
%sql
CREATE OR REPLACE TABLE silver.dim_consumidores AS
WITH deduplicated AS (
    SELECT
        customer_id,
        customer_zip_code_prefix,
        customer_name,
        customer_city,
        customer_state,
        timestamp_ingestion,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY timestamp_ingestion DESC
        ) AS rn
    FROM bronze.tb_customers
)

SELECT
    customer_id AS id_consumidor,
    customer_zip_code_prefix AS prefixo_cep,
    customer_name AS nome_consumidor,
    UPPER(customer_city) AS cidade,
    UPPER(customer_state) AS estado
FROM deduplicated
WHERE rn = 1;

num_affected_rows,num_inserted_rows


Verificando se sobrou id_consumidor repetido na tabela:

In [0]:
%sql
SELECT
    id_consumidor,
    COUNT(*) AS qtd
FROM silver.dim_consumidores
GROUP BY id_consumidor
HAVING COUNT(*) > 1;

id_consumidor,qtd


Teste para verificar se o nome de cidades e estados estão todos em maiúsculo:

In [0]:
%sql
SELECT *
FROM silver.dim_consumidores
WHERE cidade <> UPPER(cidade)
   OR estado <> UPPER(estado);

id_consumidor,prefixo_cep,nome_consumidor,cidade,estado


Teste para verificar se todas as colunas foram traduzidas:

In [0]:
%sql
DESCRIBE silver.dim_consumidores;

col_name,data_type,comment
id_consumidor,string,null
prefixo_cep,int,null
nome_consumidor,string,null
cidade,string,null
estado,string,null


Criação da silver.fat_pedidos:

In [0]:
%sql
CREATE OR REPLACE TABLE silver.fat_pedidos AS
SELECT
    order_id AS id_pedido,
    customer_id AS id_consumidor,
    order_purchase_timestamp AS data_pedido,
    order_approved_at AS data_aprovacao_pedido,
    order_delivered_carrier_date AS data_envio_transportadora,
    order_delivered_customer_date AS data_entrega_consumidor,
    order_estimated_delivery_date AS data_entrega_estimada,

    CASE order_status
        WHEN 'delivered' THEN 'entregue'
        WHEN 'canceled' THEN 'cancelado'
        WHEN 'shipped' THEN 'enviado'
        WHEN 'processing' THEN 'processando'
        WHEN 'invoiced' THEN 'faturado'
        WHEN 'unavailable' THEN 'indisponível'
        WHEN 'created' THEN 'criado'
        WHEN 'approved' THEN 'aprovado'
        ELSE 'desconhecido'
    END AS status,

    DATEDIFF(order_delivered_customer_date, order_purchase_timestamp) 
        AS tempo_entrega_dias,

    DATEDIFF(order_estimated_delivery_date, order_purchase_timestamp) 
        AS tempo_entrega_estimado_dias,

    DATEDIFF(order_delivered_customer_date, order_purchase_timestamp)
    -
    DATEDIFF(order_estimated_delivery_date, order_purchase_timestamp)
        AS diferenca_entrega_dias,

    CASE 
        WHEN order_status != 'delivered' THEN 'Não Entregue'
        WHEN order_delivered_customer_date <= order_estimated_delivery_date THEN 'Sim'
        ELSE 'Não'
    END AS entrega_no_prazo

FROM bronze.tb_orders;

num_affected_rows,num_inserted_rows


Imprimindo 10 linhas da tabela para verificar a tradução das colunas e a lógica de cálculo para as colunas adicionadas:

In [0]:
%sql
SELECT * FROM silver.fat_pedidos LIMIT 10;

id_pedido,id_consumidor,data_pedido,data_aprovacao_pedido,data_envio_transportadora,data_entrega_consumidor,data_entrega_estimada,status_pedido,tempo_entrega_dias,tempo_entrega_estimado_dias,diferenca_entrega_dias,entrega_no_prazo
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,entregue,8,16,-8,Sim
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,entregue,14,20,-6,Sim
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,entregue,9,27,-18,Sim
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,entregue,14,27,-13,Sim
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,entregue,3,13,-10,Sim
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,entregue,17,23,-6,Sim
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z,faturado,null,28,null,Não Entregue
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,entregue,10,22,-12,Sim
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z,entregue,10,42,-32,Sim
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z,entregue,18,25,-7,Sim


Verificando se os status foram traduzidos corretamente e se existiu algum que se encaixou na categoria 'desconhecido':

In [0]:
%sql
SELECT DISTINCT status
FROM silver.fat_pedidos
ORDER BY status;

status
aprovado
cancelado
criado
entregue
enviado
faturado
indisponível
processando


Criação da tabela fat_itens_pedidos:

In [0]:
%sql
CREATE OR REPLACE TABLE silver.fat_itens_pedidos AS
SELECT
    order_id       AS id_pedido,
    order_item_id  AS id_item,
    product_id     AS id_produto,
    seller_id      AS id_vendedor,
    price          AS preco_BRL,
    freight_value  AS preco_frete
FROM bronze.tb_order_items;

num_affected_rows,num_inserted_rows


Verificando se as colunas foram traduzidas corretamente:

In [0]:
%sql 
DESCRIBE silver.fat_itens_pedidos

col_name,data_type,comment
id_pedido,string,null
id_item,int,null
id_produto,string,null
id_vendedor,string,null
preco_BRL,double,null
preco_frete,double,null


Criação da tabela silver.fat_pagamentos_pedidos:

In [0]:
%sql
CREATE OR REPLACE TABLE silver.fat_pagamentos_pedidos AS
SELECT
    order_id AS id_pedido,
    payment_sequential AS sequencial_pagamento,
    payment_installments AS quantidade_parcelas,
    ROUND(payment_value,2) AS valor_pagamento,

    CASE payment_type
        WHEN 'credit_card' THEN 'Cartão de Crédito'
        WHEN 'boleto' THEN 'Boleto'
        WHEN 'voucher' THEN 'Voucher'
        WHEN 'debit_card' THEN 'Cartão de Débito'
        WHEN 'not_defined' THEN 'Não Definido'
        ELSE 'Desconhecido'
    END AS tipo_pagamento

FROM bronze.tb_order_payments;

num_affected_rows,num_inserted_rows


Verificando se os tipos de pagamento foram traduzidos corretamente e se existiu algum caso categorizado como 'desconhecido': 

In [0]:
%sql
SELECT DISTINCT tipo_pagamento
FROM silver.fat_pagamentos_pedidos
ORDER BY tipo_pagamento;

tipo_pagamento
Boleto
Cartão de Crédito
Cartão de Débito
Não Definido
Voucher


Criação da tabela silver.fat_avaliacoes_pedidos:

In [0]:
%sql
CREATE OR REPLACE TABLE silver.fat_avaliacoes_pedidos AS
WITH treated AS (
    SELECT
        review_id,
        order_id,
        review_score,

        COALESCE(review_comment_title, 'Sem título') AS review_comment_title,
        COALESCE(review_comment_message, 'Sem comentário') AS review_comment_message,

        try_to_timestamp(review_creation_date) AS review_creation_date,
        try_to_timestamp(review_answer_timestamp) AS review_answer_timestamp

    FROM bronze.tb_order_reviews
),

filtered AS (
    SELECT *
    FROM treated
    WHERE 
        order_id IS NOT NULL
        AND review_creation_date IS NOT NULL
        AND review_creation_date <= current_timestamp()
)

SELECT
    review_id AS id_avaliacao,
    order_id AS id_pedido,
    review_score AS nota_avaliacao,

    review_comment_title AS titulo_avaliacao,
    review_comment_message AS comentario_avaliacao,

    review_creation_date AS data_criacao_avaliacao,
    review_answer_timestamp AS data_resposta_avaliacao

FROM filtered;

num_affected_rows,num_inserted_rows


Verifica se existem datas no futuro:

In [0]:
%sql
SELECT *
FROM silver.fat_avaliacoes_pedidos
WHERE data_criacao_avaliacao > current_timestamp();

id_avaliacao,id_pedido,nota_avaliacao,titulo_avaliacao,comentario_avaliacao,data_criacao_avaliacao,data_resposta_avaliacao


Criação da tabela silver.dim_produtos: 

In [0]:
%sql
CREATE OR REPLACE TABLE silver.dim_produtos AS
WITH deduplicated AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY product_id
            ORDER BY timestamp_ingestion DESC
        ) AS rn
    FROM bronze.tb_products
)

SELECT
    product_id AS id_produto,
    product_name AS nome_produto,
    product_category_name AS categoria_produto,
    product_weight_g AS peso_produto_gramas,
    product_length_cm AS comprimento_centimetros,
    product_height_cm AS altura_centimetros,
    product_width_cm AS largura_centimetros,
    product_photos_qty AS quantidade_fotos,
    product_description_lenght AS tamanho_descricao_produto

FROM deduplicated
WHERE rn = 1;

num_affected_rows,num_inserted_rows


Verificando se existem id's de produto duplicados:

In [0]:
%sql
SELECT
    id_produto,
    COUNT(*) AS qtd
FROM silver.dim_produtos
GROUP BY id_produto
HAVING COUNT(*) > 1;

product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_photos_qty,product_description_lenght,timestamp_ingestion,rn,id_produto,nome_produto,categoria_produto,peso_produto_gramas,comprimento_centimetros,altura_centimetros,largura_centimetros,quantidade_fotos,tamanho_descricao_produto


Criação da tabela dim_vendedores:

In [0]:
%sql
CREATE OR REPLACE TABLE silver.dim_vendedores AS
WITH deduplicated AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY seller_id
            ORDER BY timestamp_ingestion DESC
        ) AS rn
    FROM bronze.tb_sellers
)

SELECT
    seller_id AS id_vendedor,
    seller_name AS nome_vendedor,
    seller_zip_code_prefix AS prefixo_cep,
    UPPER(seller_city) AS cidade,
    UPPER(seller_state) AS estado

FROM deduplicated
WHERE rn = 1;

num_affected_rows,num_inserted_rows


Verificando se existem id's de vendedor duplicados: 

In [0]:
%sql
SELECT
    id_vendedor,
    COUNT(*) AS qtd
FROM silver.dim_vendedores
GROUP BY id_vendedor
HAVING COUNT(*) > 1;

id_vendedor,qtd


Criação da tabela silver.dim_categoria_produtos_traducao e impressão de 20 linhas para verificação: 

In [0]:
%sql
CREATE OR REPLACE TABLE silver.dim_categoria_produtos_traducao AS
SELECT
    product_category_name        AS nome_produto_pt,
    product_category_name_english AS nome_produto_en
FROM bronze.tb_product_category_name_translation;

SELECT * FROM silver.dim_categoria_produtos_traducao LIMIT 20;

nome_produto_pt,nome_produto_en
beleza_saude,health_beauty
informatica_acessorios,computers_accessories
automotivo,auto
cama_mesa_banho,bed_bath_table
moveis_decoracao,furniture_decor
esporte_lazer,sports_leisure
perfumaria,perfumery
utilidades_domesticas,housewares
telefonia,telephony
relogios_presentes,watches_gifts


Criação da tabela silver.dim_cotacao_dolar:

In [0]:
%sql
CREATE OR REPLACE TABLE silver.dim_cotacao_dolar AS

WITH cotacao_tratada AS (
    SELECT
        TO_DATE(TRY_TO_TIMESTAMP(dataHoraCotacao)) AS data_cotacao,
        CAST(cotacaoCompra AS DOUBLE) AS valor_cotacao
    FROM bronze.tb_cotacao_dolar
    WHERE TRY_TO_TIMESTAMP(dataHoraCotacao) IS NOT NULL
),

cotacao_por_dia AS (
    SELECT
        data_cotacao,
        MAX(valor_cotacao) AS valor_cotacao
    FROM cotacao_tratada
    GROUP BY data_cotacao
),

calendario AS (
    SELECT EXPLODE(
        SEQUENCE(
            (SELECT MIN(data_cotacao) FROM cotacao_por_dia),
            (SELECT MAX(data_cotacao) FROM cotacao_por_dia),
            INTERVAL 1 DAY
        )
    ) AS data_cotacao
),

joinado AS (
    SELECT
        c.data_cotacao,
        d.valor_cotacao
    FROM calendario c
    LEFT JOIN cotacao_por_dia d
        ON c.data_cotacao = d.data_cotacao
),

preenchido AS (
    SELECT
        data_cotacao,
        LAST(valor_cotacao, TRUE) OVER (
            ORDER BY data_cotacao
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS valor_cotacao
    FROM joinado
)

SELECT
    data_cotacao,
    valor_cotacao
FROM preenchido
ORDER BY data_cotacao;

num_affected_rows,num_inserted_rows


Verificando a continuidade do calendário: 

In [0]:
%sql
SELECT
    MIN(data_cotacao) AS menor_data,
    MAX(data_cotacao) AS maior_data,
    COUNT(*) AS total_linhas,
    DATEDIFF(MAX(data_cotacao), MIN(data_cotacao)) + 1 AS total_dias_esperado
FROM silver.dim_cotacao_dolar;

menor_data,maior_data,total_linhas,total_dias_esperado
2016-01-04,2018-12-31,1093,1093


Criação da tabela silver.fat_pedido_total: 

In [0]:
%sql
CREATE OR REPLACE TABLE silver.fat_pedido_total AS

WITH pagamentos_agg AS (
    SELECT
        id_pedido,
        ROUND(SUM(valor_pagamento), 2) AS valor_total_pago_brl
    FROM silver.fat_pagamentos_pedidos
    GROUP BY id_pedido
),

pedidos AS (
    SELECT
        id_pedido,
        id_consumidor,
        status,
        CAST(data_pedido AS DATE) AS data_pedido
    FROM silver.fat_pedidos
),

cotacao AS (
    SELECT
        CAST(data_cotacao AS DATE) AS data_cotacao,
        CAST(valor_cotacao AS DOUBLE) AS valor_cotacao
    FROM silver.dim_cotacao_dolar
)

SELECT
    p.id_pedido,
    p.id_consumidor,
    p.status,
    ROUND(pg.valor_total_pago_brl, 2) AS valor_total_pago_brl,
    ROUND(
        CASE
            WHEN pg.valor_total_pago_brl IS NULL THEN NULL
            WHEN pg.valor_total_pago_brl = 0 THEN NULL
            WHEN c.valor_cotacao IS NULL THEN NULL
            WHEN c.valor_cotacao = 0 THEN NULL
            ELSE pg.valor_total_pago_brl / c.valor_cotacao
        END,
        2
    ) AS valor_total_pago_usd,
    p.data_pedido
FROM pedidos p
LEFT JOIN pagamentos_agg pg
    ON p.id_pedido = pg.id_pedido
LEFT JOIN cotacao c
    ON p.data_pedido = c.data_cotacao;

num_affected_rows,num_inserted_rows


Verifica o arredondamento de duas casas decimais: 

In [0]:
%sql
SELECT *
FROM silver.fat_pedido_total
WHERE valor_total_pago_brl != ROUND(valor_total_pago_brl, 2)
   OR (valor_total_pago_usd IS NOT NULL AND valor_total_pago_usd != ROUND(valor_total_pago_usd, 2));

id_pedido,id_consumidor,status,valor_total_pago_brl,valor_total_pago_usd,data_pedido


Otimização Física:

In [0]:
%sql
OPTIMIZE silver.fat_pedido_total
ZORDER BY (id_pedido, data_pedido);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 4282486), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1775521938918, 1775521939399, 8, 0, null, List(0, 0), null, 6, 6, 0, 0, null, null)"
